# 04 · Fase B completa — barrido de γ ∈ {0, 1, 2, 5} × semillas {42, 43, 44}
**Hipótesis H-B2 (escrita antes de correr):** la relación entre γ y la robustez cross-manipulation es no monótona; con γ = 5 el modelo ignora casi todo el train y el AUC in-domain cae. Además, este barrido dice si el resultado nulo de B-01/B-02 (Focal γ = 2 no mejora el cross-manipulation) se mantiene entre semillas.

**Protocolo:** idéntico a B-01/B-02 (D-06). Solo cambian γ y la semilla. Cada corrida guarda su JSON y `preds_<corrida>.csv.gz` (probabilidades por frame de test: solo IDs y números) en `resultados/runs/`.

**Inputs en Kaggle**
- `ffpp-crops-g8` (obligatorio).
- *(opcional)* el output del 03 o de una versión anterior de este 04: las corridas que ya tengan JSON y predicciones se copian y no se repiten. Así, si la sesión se corta, se agrega el output parcial como input y se vuelve a guardar la versión.

**Tiempo:** ~10 min por corrida en una T4 (unas 2 h para las 12). Se corre por orden de prioridad y se deja de lanzar corridas nuevas al acercarse a `MAX_HORAS`.

> **Licencia FF++:** Kaggle sube este notebook a GitHub con sus salidas. Ninguna celda muestra caras; las figuras con caras van a `privado/` (output privado de Kaggle). Tablas y gráficos numéricos sí se muestran.

In [1]:
# Kaggle: clona el repo (Internet = On). En local: usa la copia del repo desde notebooks/kaggle/
import os, sys, subprocess
from pathlib import Path
REPO_URL = "https://github.com/Gredith/DeepLearning-G8-TA_Repository.git"
ROOT = Path("/kaggle/working/repo") if Path("/kaggle").exists() else Path.cwd().parents[1]
if Path("/kaggle").exists() and not ROOT.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src" / "kaggle_e2"))
print("ROOT =", ROOT)

ROOT = /kaggle/working/repo


In [2]:
import json, shutil, time, pandas as pd, torch
from train import Config, run
import kaggle_io as K, analysis as A
CROPS = K.find_crops()
MAN = K.find_manifest(CROPS)
if MAN is not None:
    conteo = K.check_counts(CROPS, MAN); print(conteo); assert conteo.ok.all(), "Los recortes no coinciden con el manifiesto"
OUT = K.WORK / "resultados"; RUNS = OUT / "runs"; CKPT = K.WORK / "checkpoints"
RUNS.mkdir(parents=True, exist_ok=True)
BASE = dict(crops_dir=str(CROPS), out_dir=str(RUNS), ckpt_dir=str(CKPT), n_train_pairs=100, n_val_pairs=30,
            n_test_pairs=None, frames_per_video=16, epochs=6, warmup_epochs=1, batch_size=32, lr=2e-4)
if os.environ.get("SMOKE"):  # prueba rápida en CPU con datos sintéticos
    BASE.update(n_train_pairs=4, n_val_pairs=2, n_test_pairs=2, frames_per_video=3, epochs=2, batch_size=8, img_size=96, num_workers=0)
# orden de prioridad: primero confirmar BCE vs γ=2 con más semillas, luego γ=1 y γ=5
PRIORIDAD = [(0, 42), (2, 42), (0, 43), (2, 43), (0, 44), (2, 44), (1, 42), (5, 42), (1, 43), (5, 43), (1, 44), (5, 44)]
if os.environ.get("SMOKE"):
    PRIORIDAD = [(0, 42), (2, 42), (0, 43), (2, 43)]
CKPT_GUARDAR = {"bce_s42", "focal_g2_s42"}   # los que usa el notebook 06; el resto se borra para aligerar el output
MAX_HORAS = float(os.environ.get("MAX_HORAS", 10.5))   # el commit de Kaggle se corta a las 12 h
nombre = lambda g, s: "bce_s%d" % s if g == 0 else "focal_g%g_s%d" % (g, s)
print(K.snapshot_env(ROOT, OUT / "entorno.json"))

                manifiesto  en_disco    ok
Deepfakes             6394      6394  True
Face2Face             2240      2240  True
FaceSwap              2240      2240  True
NeuralTextures        2240      2240  True
original              6400      6400  True
{'commit': 'eb4eada9e620673e58b305021c54cb6fec038ec6', 'python': '3.13.16', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'gpu': 'Tesla T4', 'timm': '1.0.30', 'torchvision': '0.26.0+cu130', 'numpy': '2.1.3', 'sklearn': '1.6.1', 'facenet_pytorch': None}


## 1. Reutilizar corridas ya hechas (inputs opcionales)

In [3]:
reusadas = []
if Path("/kaggle/input").exists():
    for js in Path("/kaggle/input").rglob("*.json"):
        g, s = A.parse_run_name(js.stem)
        preds = js.parent / f"preds_{js.stem}.csv.gz"
        if g is None or not preds.exists() or (RUNS / js.name).exists():
            continue
        shutil.copy(js, RUNS / js.name); shutil.copy(preds, RUNS / preds.name); reusadas.append(js.stem)
        ck = next(Path("/kaggle/input").rglob(f"{js.stem}.pt"), None)
        if js.stem in CKPT_GUARDAR and ck is not None:
            CKPT.mkdir(parents=True, exist_ok=True); shutil.copy(ck, CKPT / ck.name)
print("reutilizadas:", sorted(reusadas) or "ninguna")

reutilizadas: ['bce_s42', 'focal_g2_s42']


## 2. Barrido

In [4]:
t0 = time.time()
for g, s in PRIORIDAD:
    name = nombre(g, s)
    if (RUNS / f"{name}.json").exists() and (RUNS / f"preds_{name}.csv.gz").exists():
        print(f"[{name}] ya existe, se salta"); continue
    horas = (time.time() - t0) / 3600
    if horas > MAX_HORAS - 0.3:
        print(f"Se detiene antes de {name}: {horas:.1f} h usadas (MAX_HORAS={MAX_HORAS}). Agregar este output como input y volver a guardar."); break
    run(Config(run_name=name, gamma=float(g), seed=s, **BASE), verbose=True)
    if name not in CKPT_GUARDAR:
        (CKPT / f"{name}.pt").unlink(missing_ok=True)
print(f"total: {(time.time() - t0) / 60:.0f} min")

[bce_s42] ya existe, se salta
[focal_g2_s42] ya existe, se salta
{
 "train": {
  "frames": 1855,
  "frames_real": 928,
  "frames_fake": 927,
  "videos": 116,
  "videos_faltantes": 284
 },
 "val": {
  "frames": 960,
  "frames_real": 480,
  "frames_fake": 480,
  "videos": 60,
  "videos_faltantes": 60
 }
}
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
[bce_s43] ep 1 (cabeza) loss=0.6807 fáciles=0.0% val_auc_frame=0.7504 val_auc_video=0.8078 (104s)
[bce_s43] ep 2 (red completa) loss=0.1881 fáciles=65.7% val_auc_frame=0.9861 val_auc_video=0.9933 (32s)
[bce_s43] ep 3 (red completa) loss=0.0467 fáciles=95.0% val_auc_frame=0.9916 val_auc_video=0.9978 (28s)
[bce_s43] ep 4 (red completa) loss=0.0295 fáciles=97.3% val_auc_frame=0.9881 val_auc_video=0.9933 (31s)
[bce_s43] ep 5 (red completa) loss=0.0249 fáciles=96.1% val_auc_frame=0.9961 val_auc_video=1.0000 (30s)
[b

## 3. Resumen (sin caras)

In [5]:
runs = A.load_runs(RUNS)
tabla = A.metrics_table(runs); tabla.to_csv(OUT / "barrido_metricas.csv", index=False)
resumen = A.summary_by_gamma(tabla, "auc_video"); resumen.to_csv(OUT / "barrido_resumen_auc_video.csv", index=False)
print(f"{len(runs)} corridas:", sorted(runs))
resumen.pivot(index="test", columns="gamma", values="texto")

12 corridas: ['bce_s42', 'bce_s43', 'bce_s44', 'focal_g1_s42', 'focal_g1_s43', 'focal_g1_s44', 'focal_g2_s42', 'focal_g2_s43', 'focal_g2_s44', 'focal_g5_s42', 'focal_g5_s43', 'focal_g5_s44']


gamma,0.0,1.0,2.0,5.0
test,,,,
Deepfakes,0.991 ± 0.007,0.994 ± 0.006,0.993 ± 0.005,0.988 ± 0.010
Face2Face,0.657 ± 0.028,0.653 ± 0.019,0.660 ± 0.024,0.651 ± 0.035
FaceSwap,0.265 ± 0.030,0.240 ± 0.024,0.229 ± 0.036,0.300 ± 0.082
NeuralTextures,0.758 ± 0.004,0.783 ± 0.013,0.783 ± 0.037,0.758 ± 0.039


Para llevar los resultados al repo: descargar `resultados/` del output (JSON, `preds_*.csv.gz`, CSV, `entorno.json`, `pip_freeze.txt`) y copiarlos a `results/metrics/`. Los checkpoints quedan en Kaggle; el notebook 06 usa este output como input.